In [1]:
pip install --upgrade selenium

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
import requests
import pandas as pd

## Use this to remove emojis if there's any
def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  
        "\U0001F300-\U0001F5FF"  
        "\U0001F680-\U0001F6FF"  
        "\U0001F700-\U0001F77F"  
        "\U0001F780-\U0001F7FF"  
        "\U0001F800-\U0001F8FF"  
        "\U0001F900-\U0001F9FF"  
        "\U0001FA00-\U0001FA6F"  
        "\U0001FA70-\U0001FAFF"  
        "\U00002702-\U000027B0"  
        "\U000024C2-\U0001F251" 
        "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

article_dict = {}
article_links = {}

driver = webdriver.Chrome()


driver.get(r'https://www.backscoop.com/')
time.sleep(2) 
    
ActionChains(driver).send_keys(Keys.PAGE_DOWN).perform()
time.sleep(2) 
    
driver.find_element(By.CLASS_NAME, "cta-button-large.w-button").click()
time.sleep(2) 

html = driver.page_source
soup = BeautifulSoup(html, 'html.parser') 
article_list = soup.find_all("a", class_= "flex-align-start w-inline-block")

for article in article_list:
    link = r'https://www.backscoop.com/' + article.get('href')
    driver.get(link)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    title = soup.find('h1', class_="h2---l black-text mobile-h2-l").get_text()
    content = soup.find_all("p")
    article_text = "\n".join([p.get_text() for p in content])
    article_dict[title] = article_text.replace("The newsletter that keeps you up-to-date on the top stories on tech and business in Southeast Asia. It's fun, quick and free.", "")
    article_links[title] = link
article_dict = {title: remove_emojis(text) for title, text in article_dict.items()}


driver.get('https://www.aseanbriefing.com/news/')
time.sleep(2) 

html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

news = soup.find_all('div', class_="content news mb-2rem")
article_list = []
for entry in news:
     link = entry.find('a')
     news_article = link.get('href')
     article_list.append(news_article)

for article in article_list:
     driver.get(article)
     time.sleep(2)
     html = driver.page_source
     soup = BeautifulSoup(html, 'html.parser')
     title = soup.find('h1', class_="title-border").get_text()
     content = soup.find_all("p")
     DSA_event_block = soup.find('div', class_="dsa-events-block")
     section_ender = soup.find('section', class_ = 'article-section dsa-cta mt-5rem')
     firm_credits = soup.find('div', class_="sidebox download-card dsa-cta-rhs mb-3rem mt-3rem")
     subs_marketing = soup.find('div', class_="dsa-ab-weekly mt-3rem")
     article_credit = soup.find('div', class_="article-credit")
     sign_up = soup.find('div', class_='sign__upform-lhs')
     search_bar = soup.find('div', class_='result_step1 countryGuideSearchPopup')
    
     exclusion_blocks = [DSA_event_block, section_ender, firm_credits, subs_marketing, article_credit,sign_up]
     exclusion_paragraphs = []
    
     for block in exclusion_blocks:
         if block:
             exclusion_paragraphs.extend(block.find_all('p'))
    
     article_text = "\n".join([p.get_text() for p in content if p not in exclusion_paragraphs])
     article_dict[title] = article_text
     article_links[title] = article

driver.get('https://asianbankingandfinance.net/market/southeast-asia')
time.sleep(2) 

html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

main_content_div = soup.find('div', class_='view-content')

if main_content_div:
     news = soup.find_all('h2', class_="item__title size-24")
     article_list = []
     for entry in news:
         link = entry.find('a')
         news_article = link.get('href')
         article_list.append(news_article)

for article in article_list:
    driver.get(article)
    time.sleep(2)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    title = soup.find('h1', class_="nf__title page-header text-black size-37 striptags").get_text()
    title = title.strip()
    content = soup.find_all("p")
    section_ender = soup.find('section', class_ = 'block block-story-here mb-20')
    firm_credits = soup.find('div', class_= "d-none d-lg-block")
    footer_credits = soup.find('div', class_ = 'footer-bottom text-center text-white')
    wrapper_plug = soup.find('div', class_ ='wrapper')
    exclusion_blocks = [section_ender, firm_credits, footer_credits, wrapper_plug]
    exclusion_paragraphs = []
    
    for block in exclusion_blocks:
        if block:
           exclusion_paragraphs.extend(block.find_all('p'))
    
    article_text = "\n".join([p.get_text() for p in content if p not in exclusion_paragraphs and "ALSO READ" not in p.get_text()])
    article_dict[title] = article_text
    article_links[title] = article

driver.get('https://www.xe.com/currencyconverter/convert/?Amount=1&From=USD&To=PHP')
time.sleep(2)  

rate_element = driver.find_element(By.XPATH, '//*[@id="__next"]/div[4]/div[2]/section/div[2]/div/main/div/div[2]/div[1]/div/p[2]')
rate_text = rate_element.text

rate_match = re.search(r'([\d.]+)', rate_text)
if rate_match:
    rate = round(float(rate_match.group(1)), 2)
else:
    rate = None

update_element = driver.find_element(By.XPATH, '//*[@id="__next"]/div[4]/div[2]/section/div[2]/div/main/div/div[2]/div[3]/div[2]/div[2]')
update_text = update_element.text

match = re.search(r'Last updated (.+)', update_text)
if match:
    date_time_str = match.group(1)
    datetime_obj = pd.to_datetime(date_time_str)

    datetime_obj += pd.Timedelta(hours = 8)

    last_updated = datetime_obj.strftime("%H:%M of %B %d, %Y")
else:
    last_updated = "Date and time not found"

url = 'https://frames.pse.com.ph/stockTable/mostActive'
response = requests.get(url)

if response.status_code == 200:
    soup = BeautifulSoup(response.content, "html.parser")
    table = soup.find("table", class_="table table-hover border table-resizable")
    if table:
        top_entries = []
        rows = table.find("tbody").find_all("tr")
        #You can change how many entries you want here
        for row in rows[:3]:
            cells = row.find_all("td")
            if len(cells) >= 5:
                entry = f"{cells[0].text.strip()}. {cells[1].text.strip()} - Last Price: {cells[2].text.strip()}, Change: {cells[3].text.strip()}, % Change: {cells[4].text.strip()}"
                top_entries.append(entry)
driver.quit()

print(rate, last_updated)

58.37 09:09 of July 19, 2024


In [32]:
print(top_entries)

['Ayala Land, Inc.. ALI - Last Price: 30.95, Change: -0.70, % Change: -2.21%', 'Bank of the Philippine Islands. BPI - Last Price: 126.00, Change: 5.20, % Change: 4.30%', 'BDO Unibank, Inc.. BDO - Last Price: 142.50, Change: 3.10, % Change: 2.22%']


In [48]:
# Extract the relevant parts
def extract_stock_info(stock_info):
    parts = stock_info.split(' - ')
    name_code = parts[0].rsplit('. ', 1)  # Split by the last occurrence of '. '
    name = f"{name_code[0]} ({name_code[1]})"
    change = parts[1].split(', ')[1].split(': ')[1]
    percent_change = parts[1].split(', ')[2].split(': ')[1]
    return name, change, percent_change

# Extract information for each stock
stock1_name, stock1_change, stock1_percent_change = extract_stock_info(top_entries[0])
stock2_name, stock2_change, stock2_percent_change = extract_stock_info(top_entries[1])
stock3_name, stock3_change, stock3_percent_change = extract_stock_info(top_entries[2])

# Helper function to determine the wording
def format_change(change):
    change_value = float(change)
    if change_value > 0:
        return f"up {change_value}"
    else:
        return f"down {abs(change_value)}"

def format_percent_change(percent_change):
    percent_value = float(percent_change.strip('%'))
    if percent_value > 0:
        return f"an increase of {percent_value}%"
    else:
        return f"a decrease of {abs(percent_value)}%"

# Create the f-string with the relevant information
f_string = f"""
As of today, the Philippine peso is holding steady against the dollar at {rate} as of {last_updated}. 

In the Philippine Stock Exchange, the top 3 most active stocks are driving the market with their impressive performances: 
{stock1_name} has seen a real change of {format_change(stock1_change)} with {format_percent_change(stock1_percent_change)}, 
{stock2_name} shows a real change of {format_change(stock2_change)} with {format_percent_change(stock2_percent_change)}, 
and {stock3_name} is {format_change(stock3_change)} with {format_percent_change(stock3_percent_change)}. 

Let's dive into some of the exciting developments in the market!
"""

print(f_string)


As of today, the Philippine peso is holding steady against the dollar at 58.37 as of 09:09 of July 19, 2024. 

In the Philippine Stock Exchange, the top 3 most active stocks are driving the market with their impressive performances: 
Ayala Land, Inc. (ALI) has seen a real change of down 0.7 with a decrease of 2.21%, 
Bank of the Philippine Islands (BPI) shows a real change of up 5.2 with an increase of 4.3%, 
and BDO Unibank, Inc. (BDO) is up 3.1 with an increase of 2.22%. 

Let's dive into some of the exciting developments in the market!



### Dictionary Article Contents Preview

##### if the emoji is still there the unicode for that emoji has not been recorded in python pa i think

In [67]:
for content in article_dict.values():
    print(content)
    print("-------")

Greg Krasnov is the Founder and CEO of Tonik, the first digital bank in the Philippines.  Since launching 3 years ago they have onboarded over 1.5M clients, and are rapidly growing their loan portfolio. They’ve raised $160M raised to date, and Tonik is his fifth startup.
‍
  How would you explain your job to someone outside tech?
I run the first digital bank in the Philippines. No branches, app only. We do loans, savings, and payments.
‍
  What's something about you or your job that would surprise us?
I spent 2 years cruising Southeast Asia on a sailboat during a sabbatical.
‍
  What has been the biggest highlight of your career so far?
Launching the first digital bank in the Philippines. We are solving a massive credit access problem, whereby 90% of Filipinos have never had a bank loan. Huge potential for upward mobility of the Filipino population.
‍
  What's a startup trend or space you're watching this year?
Gen AI, like everyone. Massive potential to completely revolutionize how hu

In [14]:
import os
import json
from openai import OpenAI
import tiktoken

# Ensure the API key environment variable is set
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise ValueError("The OpenAI API key must be set in the 'OPENAI_API_KEY' environment variable.")

client = OpenAI(api_key=api_key)

def split_data(data, token_limit, tokenizer, prompt_token_count):
    chunks = []
    current_chunk = {}
    current_chunk_length = prompt_token_count  # Start with the prompt token count

    for key, value in data.items():
        key_value_str = json.dumps({key: value})
        key_value_tokens = tokenizer.encode(key_value_str)
        key_value_token_count = len(key_value_tokens)

        if current_chunk_length + key_value_token_count > token_limit:
            chunks.append(current_chunk)
            current_chunk = {}
            current_chunk_length = prompt_token_count  # Reset to prompt token count

        current_chunk[key] = value
        current_chunk_length += key_value_token_count

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

def send_chunks(prompt_template, article_chunks, chat_model="gpt-3.5-turbo", token_limit=8192):
    responses = []
    tokenizer = tiktoken.encoding_for_model(chat_model)
    prompt_tokens = tokenizer.encode(prompt_template)
    prompt_token_count = len(prompt_tokens)

    # Token limit adjusted to ensure it accommodates both prompt and data
    chunk_size = token_limit - prompt_token_count

    for chunk in article_chunks:
        chunk_json = json.dumps(chunk)
        chunk_tokens = tokenizer.encode(chunk_json)
        chunk_token_count = len(chunk_tokens)
        
        if chunk_token_count > chunk_size:
            raise ValueError("Chunk exceeds the allowed token limit.")

        prompt = f"{prompt_template}\nArticles:\n{chunk_json}"
        messages = [{"role": "user", "content": prompt}]

        response = client.chat.completions.create(model=chat_model, messages=messages)
        chatgpt_response = response.choices[0].message.content.strip()
        responses.append(chatgpt_response)

    return responses

articles = article_dict

prompt_template = """
I have a Python dictionary containing articles, and I need concise summaries for each article tailored for a business student in Southeast Asia who is fascinated by the startup ecosystem. Each summary should be less than 150 words, maintaining nuance and focusing on salient points:

**Introduction:**
Identify the article's main subject and its relevance to the startup ecosystem in Southeast Asia.

**Key Information:**
1. **Company/Product:** Summarize the company's or product's main focus.
2. **Achievements/Funding:** Note any recent achievements or funding rounds.
3. **Problem and Solution:** Highlight the problem the company addresses and its solution.
4. **Future Plans:** Mention any future plans or upcoming developments.

**Context:**
Provide context on how this information fits into broader trends in the Southeast Asian startup ecosystem.

**Relevance:**
Explain why this information is particularly relevant or interesting to business students in the region.

Summaries must be returned in the following format as a Python Dictionary of type dict:

"Article Title": {
"Introduction": "",
"Key Information": {
"Company/Product": "",
"Achievements/Funding": "",
"Problem and Solution": "",
"Future Plans": ""
},
"Context": "",
"Relevance": ""
},
"""

# Tokenizer for splitting the articles
tokenizer = tiktoken.encoding_for_model("gpt-3.5-turbo")

# Calculate the token count for the prompt
prompt_tokens = tokenizer.encode(prompt_template)
prompt_token_count = len(prompt_tokens)

# Adjust token limit for the actual model
token_limit = 8192  # Adjust based on your model

# Split the articles into chunks based on the token limit
article_chunks = split_data(articles, token_limit, tokenizer, prompt_token_count)

# Send the chunks to the OpenAI API and get responses
summary_responses = send_chunks(prompt_template, article_chunks, token_limit=token_limit)

# Combine all the summary responses into a single dictionary
combined_summaries = {}
for response in summary_responses:
    try:
        partial_summary = json.loads(response)
        if isinstance(partial_summary, dict):
            combined_summaries.update(partial_summary)
        else:
            print(f"Warning: Non-dictionary response encountered: {response}")
    except json.JSONDecodeError as e:
        print(f"Error parsing response: {e}")

# Print the combined summaries
print("Summary for all articles:")
print(json.dumps(combined_summaries, indent=2))

Summary for all articles:
{
  "Ice Breakers with Greg Krasnov (Founder and CEO, Tonik)": {
    "Introduction": "Greg Krasnov is the Founder and CEO of Tonik, the first digital bank in the Philippines, making waves in the Southeast Asian startup ecosystem.",
    "Key Information": {
      "Company/Product": "Tonik, the first digital bank in the Philippines, focusing on loans, savings, and payments.",
      "Achievements/Funding": "$160M raised to date, with over 1.5M clients onboarded.",
      "Problem and Solution": "Addressing the credit access problem in the Philippines where 90% of Filipinos have never had a bank loan, providing opportunities for upward mobility.",
      "Future Plans": "Looking into Gen AI technology to revolutionize human interactions with technology."
    },
    "Context": "Tonik's success reflects the growing digital banking trend in Southeast Asia, where innovative solutions are reshaping traditional banking models.",
    "Relevance": "Business students in Sout

In [16]:
combined_summaries

{'Ice Breakers with Greg Krasnov (Founder and CEO, Tonik)': {'Introduction': 'Greg Krasnov is the Founder and CEO of Tonik, the first digital bank in the Philippines, making waves in the Southeast Asian startup ecosystem.',
  'Key Information': {'Company/Product': 'Tonik, the first digital bank in the Philippines, focusing on loans, savings, and payments.',
   'Achievements/Funding': '$160M raised to date, with over 1.5M clients onboarded.',
   'Problem and Solution': 'Addressing the credit access problem in the Philippines where 90% of Filipinos have never had a bank loan, providing opportunities for upward mobility.',
   'Future Plans': 'Looking into Gen AI technology to revolutionize human interactions with technology.'},
  'Context': "Tonik's success reflects the growing digital banking trend in Southeast Asia, where innovative solutions are reshaping traditional banking models.",
  'Relevance': "Business students in Southeast Asia can learn from Tonik's innovative approach to digi

In [54]:
import os
import json
from notion_client import Client
from datetime import datetime
import random

# List of emojis to choose from
icons = ["🫨", "😶", "😐", "🫥", "😶‍🌫️", "😬", "😑"]

# Choose a random icon based on the length of children_blocks
random_icon = random.choice(icons)

# Initialize Notion client with your API token
notion = Client(auth='secret_QdHuRpFVjlzo0duFXRDJv0QQkffWVuvvEDSATdFVR31')

# Define the database ID where the page will be created
database_id = 'f6e1f43bfd814da782cb80d29862fb34'

# Get the latest summary response
articles = combined_summaries

# Ensure articles is a dictionary
if isinstance(articles, str):
    raise ValueError("Expected articles to be a dictionary, got string instead.")

# Function to create the content blocks
def create_content_blocks(article_title, article_content):
    children_blocks = []

    def find_matching_link(article_title):
        for key, link in article_links.items():
            if key == article_title:
                return link
        return None

    link = find_matching_link(article_title)
    
    # Add heading 1 for the article title
    heading_block = {
        "object": "block",
        "type": "heading_1",
        "heading_1": {
            "rich_text": [{
                "type": "text",
                "text": {"content": article_title},
                "annotations": {
                    "color": "yellow",
                    "underline": True
                }
            }]
        }
    }
    
    # If link exists, add it to the heading block
    if link:
        heading_block["heading_1"]["rich_text"][0]["text"]["link"] = {
            "type": "url",
            "url": link
        }
    
    children_blocks.append(heading_block)

    for section, content in article_content.items():
        if section == "Introduction":
            # Add content under 'Introduction' directly as a paragraph
            children_blocks.append({
                "object": "block",
                "type": "paragraph",
                "paragraph": {
                    "rich_text": [{"type": "text", "text": {"content": content}}]
                }
            })
        elif section == "Key Information":
            # Add sub-sections for 'Key Information' as heading 3 sections
            for key, value in content.items():
                children_blocks.append({
                    "object": "block",
                    "type": "heading_3",
                    "heading_3": {
                        "rich_text": [{
                            "type": "text",
                            "text": {"content": key},
                            "annotations": {
                                "underline": True
                            }
                        }]
                    }
                })
                
                children_blocks.append({
                    "object": "block",
                    "type": "paragraph",
                    "paragraph": {
                        "rich_text": [{"type": "text", "text": {"content": value}}]
                    }
                })
        else:
            # Add heading for other sections
            children_blocks.append({
                "object": "block",
                "type": "heading_2",
                "heading_2": {
                    "rich_text": [{
                        "type": "text",
                        "text": {"content": section},
                        "annotations": {
                            "color": "yellow"
                        }
                    }]
                }
            })

            if isinstance(content, dict):
                # Add sub-sections for other sections
                for sub_section, sub_content in content.items():
                    children_blocks.append({
                        "object": "block",
                        "type": "heading_3",
                        "heading_3": {
                            "rich_text": [{"type": "text", "text": {"content": sub_section}}]
                        }
                    })
                    children_blocks.append({
                        "object": "block",
                        "type": "paragraph",
                        "paragraph": {
                            "rich_text": [{"type": "text", "text": {"content": sub_content}}]
                        }
                    })
            else:
                # Add text content for other sections
                children_blocks.append({
                    "object": "block",
                    "type": "paragraph",
                    "paragraph": {
                        "rich_text": [{"type": "text", "text": {"content": content}}]
                    }
                })

    return children_blocks

# Create the page in the database
def create_notion_page():
    current_date = datetime.now().strftime("%Y-%m-%d")
    
    # Define the maximum number of blocks per column (adjust as needed)
    max_blocks_per_column = 100
    
    # Prepare the columns
    columns = [[]]
    current_column_index = 0

    for article_title, article_content in articles.items():
        article_blocks = create_content_blocks(article_title, article_content)
        
        # Check if the article fits in the current column
        if len(columns[current_column_index]) + len(article_blocks) > max_blocks_per_column:
            # Start a new column if the current one exceeds the limit
            columns.append([])
            current_column_index += 1

        # Add article blocks to the current column
        columns[current_column_index].extend(article_blocks)

    # Create the column list
    column_blocks = [
        {
            "object": "block",
            "type": "column_list",
            "column_list": {
                "children": [
                    {
                        "object": "block",
                        "type": "column",
                        "column": {
                            "children": column
                        }
                    } for column in columns
                ]
            }
        }
    ]
    
    # Create the callout block with f-string content
    callout_block = {
        "object": "block",
        "type": "callout",
        "callout": {
            "rich_text": [
                {"type": "text", "text": {"content": "As of today, the Philippine peso is holding steady against the dollar at "}},
                {"type": "text", "text": {"content": str(rate)}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": " as of "}},
                {"type": "text", "text": {"content": str(last_updated)}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": ". In the Philippine Stock Exchange, the top 3 most active stocks are driving the market with their impressive performances: "}},
                {"type": "text", "text": {"content": stock1_name}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": " has seen a real change of "}},
                {"type": "text", "text": {"content": format_change(stock1_change)}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": " with "}},
                {"type": "text", "text": {"content": format_percent_change(stock1_percent_change)}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": ", "}},
                {"type": "text", "text": {"content": stock2_name}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": " shows a real change of "}},
                {"type": "text", "text": {"content": format_change(stock2_change)}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": " with "}},
                {"type": "text", "text": {"content": format_percent_change(stock2_percent_change)}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": ", and "}},
                {"type": "text", "text": {"content": stock3_name}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": " is "}},
                {"type": "text", "text": {"content": format_change(stock3_change)}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": " with "}},
                {"type": "text", "text": {"content": format_percent_change(stock3_percent_change)}, "annotations": {"bold": True, "color": "yellow", "underline": True}},
                {"type": "text", "text": {"content": ". Let's dive into some of the exciting developments in the market!"}}
            ],
            "icon": {
                "emoji": "⭐"
            },
            "color": "default"
        }
    }

    # Add the callout block at the top of the children blocks
    children_blocks = [callout_block] + column_blocks

    # Create the page in the database
    new_page = notion.pages.create(
        parent={"database_id": database_id},
        icon={
            "type": "emoji",
            "emoji": random_icon
        },
        properties={
            "Name": {
                "title": [
                    {
                        "type": "text",
                        "text": {
                            "content": current_date
                        }
                    }
                ]
            }
        },
        children=children_blocks
    )
    
    return new_page

def extract_stock_info(stock_info):
    parts = stock_info.split(' - ')
    name_code = parts[0].rsplit('. ', 1)  # Split by the last occurrence of '. '
    name = f"{name_code[0]} ({name_code[1]})"
    change = parts[1].split(', ')[1].split(': ')[1]
    percent_change = parts[1].split(', ')[2].split(': ')[1]
    return name, change, percent_change

# Extract information for each stock
stock1_name, stock1_change, stock1_percent_change = extract_stock_info(top_entries[0])
stock2_name, stock2_change, stock2_percent_change = extract_stock_info(top_entries[1])
stock3_name, stock3_change, stock3_percent_change = extract_stock_info(top_entries[2])

# Helper function to determine the wording
def format_change(change):
    change_value = float(change)
    if change_value > 0:
        return f"up {change_value}"
    else:
        return f"down {abs(change_value)}"

def format_percent_change(percent_change):
    percent_value = float(percent_change.strip('%'))
    if percent_value > 0:
        return f"an increase of {percent_value}%"
    else:
        return f"a decrease of {abs(percent_value)}%"

# Create the Notion page with the articles
create_notion_page()

print("Page created successfully!")

Page created successfully!


### SCRATCH CELL

In [7]:
#driver.get(r'https://www.asiafinancial.com/regions/southeast-asia')
#time.sleep(2) 

#html = driver.page_source
#soup = BeautifulSoup(html, 'html.parser') 
#article_list = soup.find_all("a", class_= "tt-post-title c-h5")

#for article in article_list:
#    link = article.get('href')
#    driver.get(link)
#    time.sleep(2) 
#    html = driver.page_source
#    soup = BeautifulSoup(html, 'html.parser')
#    title = soup.find('h1', class_="reports-big-head").get_text()
#    content = soup.find_all("p")
#    author_section = soup.find('div', class_="author-content")
#    footer_section = soup.find('div', class_="col-md-3 col-sm-6 col-1")
#    tagline_section = soup.find('div', class_="kl-private-reset-css-Xuajs1 go3176171171")
#    article_text = ''
#    for p in content:
#        if (author_section is None or p not in author_section.find_all("p")) and \
#           (footer_section is None or p not in footer_section.find_all("p")) and \
#           (tagline_section is None or p not in tagline_section.find_all("p")):
#           article_text += p.get_text() + "\n"
#    article_dict[title] = article_text
#    article_links[title] = link

In [99]:

import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
article_dict = {}
driver = webdriver.Chrome()

driver.get('https://asianbankingandfinance.net/market/southeast-asia')
time.sleep(2) 

html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

main_content_div = soup.find('div', class_='view-content')

if main_content_div:
    news = soup.find_all('h2', class_="item__title size-24")
    article_list = []
    for entry in news:
        link = entry.find('a')
        news_article = link.get('href')
        article_list.append(news_article)

for article in article_list:
    driver.get(article)
    time.sleep(2)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    title = soup.find('h1', class_="nf__title page-header text-black size-37 striptags").get_text()
    title = title.strip()
    content = soup.find_all("p")
    section_ender = soup.find('section', class_ = 'block block-story-here mb-20')
    firm_credits = soup.find('div', class_= "d-none d-lg-block")
    footer_credits = soup.find('div', class_ = 'footer-bottom text-center text-white')
    exclusion_blocks = [section_ender, firm_credits, footer_credits]
    exclusion_paragraphs = []
    
    for block in exclusion_blocks:
        if block:
           exclusion_paragraphs.extend(block.find_all('p'))
    
    article_text = "\n".join([p.get_text() for p in content if p not in exclusion_paragraphs and "ALSO READ" not in p.get_text()])
    article_dict[title] = article_text
    
driver.quit()

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=126.0.6478.127)
Stacktrace:
	GetHandleVerifier [0x00007FF7D5F8EEA2+31554]
	(No symbol) [0x00007FF7D5F07ED9]
	(No symbol) [0x00007FF7D5DC872A]
	(No symbol) [0x00007FF7D5D9D995]
	(No symbol) [0x00007FF7D5E444D7]
	(No symbol) [0x00007FF7D5E5C051]
	(No symbol) [0x00007FF7D5E3CDD3]
	(No symbol) [0x00007FF7D5E0A33B]
	(No symbol) [0x00007FF7D5E0AED1]
	GetHandleVerifier [0x00007FF7D6298B1D+3217341]
	GetHandleVerifier [0x00007FF7D62E5AE3+3532675]
	GetHandleVerifier [0x00007FF7D62DB0E0+3489152]
	GetHandleVerifier [0x00007FF7D603E776+750614]
	(No symbol) [0x00007FF7D5F1375F]
	(No symbol) [0x00007FF7D5F0EB14]
	(No symbol) [0x00007FF7D5F0ECA2]
	(No symbol) [0x00007FF7D5EFE16F]
	BaseThreadInitThunk [0x00007FFF121B257D+29]
	RtlUserThreadStart [0x00007FFF1304AF28+40]


In [21]:
pip install tiktoken

   ---------------------------------------- 0.0/799.0 kB ? eta -:--:--
    --------------------------------------- 10.2/799.0 kB ? eta -:--:--
   - ------------------------------------- 30.7/799.0 kB 262.6 kB/s eta 0:00:03
   - ------------------------------------- 30.7/799.0 kB 262.6 kB/s eta 0:00:03
   -- ------------------------------------ 61.4/799.0 kB 328.2 kB/s eta 0:00:03
   ----- -------------------------------- 122.9/799.0 kB 516.7 kB/s eta 0:00:02
   --------- ---------------------------- 204.8/799.0 kB 778.2 kB/s eta 0:00:01
   ----------- -------------------------- 245.8/799.0 kB 795.7 kB/s eta 0:00:01
   ----------------- ---------------------- 358.4/799.0 kB 1.0 MB/s eta 0:00:01
   ------------------ ------------------- 389.1/799.0 kB 971.5 kB/s eta 0:00:01
   ------------------------ --------------- 491.5/799.0 kB 1.1 MB/s eta 0:00:01
   -------------------------- ------------- 522.2/799.0 kB 1.1 MB/s eta 0:00:01
   -------------------------- ------------- 522.2/799.0 k

In [97]:
article_dict

{"UOB poised to benefit from growth of SEA's mass affluent segment": 'Article content here...',
 'APAC Islamic banks continue upward trajectory with lift from new players': 'Article content here...',
 'Why BNPL is booming in Southeast Asia': 'Article content here...'}

In [12]:
import tiktoken as tk
enc = tk.get_encoding("o200k_base")

# To get the tokeniser corresponding to a specific model in the OpenAI API:
enc = tk.encoding_for_model("gpt-4o")
encoded = enc.encode(str(article_dict))
len(encoded)

2213

In [9]:
pip install notion-client

Note: you may need to restart the kernel to use updated packages.
